# UK Used Car Price Prediction: 01. Data Audit dan Cleaning

Notebook ini memuat, mengaudit, dan membersihkan dataset *100,000 UK Used Car Data set* (Kaggle) sebelum dipakai untuk pemodelan.

**Output:** `data/processed/cars_clean.csv` (dipakai notebook berikutnya).

**Cara menjalankan:** pilih kernel `.venv`, lalu **Run All** dari atas. Letakkan file CSV dataset di `data/raw/`.

## 1. Setup dan Load Data

In [11]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)

# notebook ada di folder notebooks/, jadi data ada satu level di atas
DATA_RAW = Path("../data/raw")
print(sorted(p.name for p in DATA_RAW.glob("*.csv")))

['audi.csv', 'bmw.csv', 'cclass.csv', 'focus.csv', 'ford.csv', 'hyundi.csv', 'merc.csv', 'skoda.csv', 'toyota.csv', 'unclean cclass.csv', 'unclean focus.csv', 'vauxhall.csv', 'vw.csv']


In [12]:
for f in sorted(DATA_RAW.glob("*.csv")):
    tmp = pd.read_csv(f)
    print(f"{f.name:22} {str(tmp.shape):12} {list(tmp.columns)}")

audi.csv               (10668, 9)   ['model', 'year', 'price', 'transmission', 'mileage', 'fuelType', 'tax', 'mpg', 'engineSize']
bmw.csv                (10781, 9)   ['model', 'year', 'price', 'transmission', 'mileage', 'fuelType', 'tax', 'mpg', 'engineSize']
cclass.csv             (3899, 7)    ['model', 'year', 'price', 'transmission', 'mileage', 'fuelType', 'engineSize']
focus.csv              (5454, 7)    ['model', 'year', 'price', 'transmission', 'mileage', 'fuelType', 'engineSize']
ford.csv               (17965, 9)   ['model', 'year', 'price', 'transmission', 'mileage', 'fuelType', 'tax', 'mpg', 'engineSize']
hyundi.csv             (4860, 9)    ['model', 'year', 'price', 'transmission', 'mileage', 'fuelType', 'tax(£)', 'mpg', 'engineSize']
merc.csv               (13119, 9)   ['model', 'year', 'price', 'transmission', 'mileage', 'fuelType', 'tax', 'mpg', 'engineSize']
skoda.csv              (6267, 9)    ['model', 'year', 'price', 'transmission', 'mileage', 'fuelType', 'tax', 'mpg',

In [13]:
def read_clean(path):
    """Baca satu CSV, rapikan nama kolom dan teks."""
    d = pd.read_csv(path)
    d.columns = d.columns.str.strip()
    d = d.rename(columns={"tax(£)": "tax"})   # nama kolom beda di sebagian file
    for col in d.select_dtypes(include=["object", "string"]).columns:
        d[col] = d[col].str.strip()
    return d

# Hanya 9 file merek utama (cclass, focus, dan file unclean dikeluarkan:
# lihat pengecekan overlap di bagian 3)
BRANDS = ["audi", "bmw", "ford", "hyundi", "merc",
          "skoda", "toyota", "vauxhall", "vw"]

frames = []
for b in BRANDS:
    d = read_clean(DATA_RAW / f"{b}.csv")
    d["brand"] = b
    frames.append(d)

df = pd.concat(frames, ignore_index=True)
print(df.shape)
print(df.columns.tolist())
df.head()

(99187, 10)
['model', 'year', 'price', 'transmission', 'mileage', 'fuelType', 'tax', 'mpg', 'engineSize', 'brand']


,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,brand
0,A1,2017,12500,Manual,15735,Petrol,150,55.4,1.4,audi
1,A6,2016,16500,Automatic,36203,Diesel,20,64.2,2.0,audi
2,A1,2016,11000,Manual,29946,Petrol,30,55.4,1.4,audi
3,A4,2017,16800,Automatic,25952,Diesel,145,67.3,2.0,audi
4,A3,2019,17300,Manual,1998,Petrol,145,49.6,1.0,audi


## 2. Audit Kualitas Data

In [14]:
df.info()
print("\nNilai kosong per kolom:")
print(df.isna().sum())
print("\nBaris duplikat:", df.duplicated().sum())
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 99187 entries, 0 to 99186
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   model         99187 non-null  str    
 1   year          99187 non-null  int64  
 2   price         99187 non-null  int64  
 3   transmission  99187 non-null  str    
 4   mileage       99187 non-null  int64  
 5   fuelType      99187 non-null  str    
 6   tax           99187 non-null  int64  
 7   mpg           99187 non-null  float64
 8   engineSize    99187 non-null  float64
 9   brand         99187 non-null  str    
dtypes: float64(2), int64(4), str(4)
memory usage: 7.6 MB

Nilai kosong per kolom:
model           0
year            0
price           0
transmission    0
mileage         0
fuelType        0
tax             0
mpg             0
engineSize      0
brand           0
dtype: int64

Baris duplikat: 1475


,year,price,mileage,tax,mpg,engineSize
count,99187.000000,99187.000000,99187.000000,99187.000000,99187.000000,99187.000000
mean,2017.087723,16805.347656,23058.914213,120.299838,55.166825,1.663280
std,2.123934,9866.773417,21148.523721,63.150926,16.138522,0.557646
min,1970.000000,450.000000,1.000000,0.000000,0.300000,0.000000
25%,2016.000000,9999.000000,7425.000000,125.000000,47.100000,1.200000
50%,2017.000000,14495.000000,17460.000000,145.000000,54.300000,1.600000
75%,2019.000000,20870.000000,32339.000000,145.000000,62.800000,2.000000
max,2060.000000,159999.000000,323000.000000,580.000000,470.800000,6.600000


In [15]:
print("year min/max      :", df["year"].min(), "/", df["year"].max())
print("engineSize == 0   :", (df["engineSize"] == 0).sum())
print("price min/max     :", df["price"].min(), "/", df["price"].max())
print("mileage min/max   :", df["mileage"].min(), "/", df["mileage"].max())
print()
for col in ["transmission", "fuelType", "brand"]:
    print(df[col].value_counts(dropna=False), "\n")

year min/max      : 1970 / 2060
engineSize == 0   : 273
price min/max     : 450 / 159999
mileage min/max   : 1 / 323000

transmission
Manual       56445
Semi-Auto    22677
Automatic    20056
Other            9
Name: count, dtype: int64 

fuelType
Petrol      54928
Diesel      40928
Hybrid       3078
Other         247
Electric        6
Name: count, dtype: int64 

brand
ford        17965
vw          15157
vauxhall    13632
merc        13119
bmw         10781
audi        10668
toyota       6738
skoda        6267
hyundi       4860
Name: count, dtype: int64 



## 3. Inspeksi Anomali

In [16]:
key = ["model", "year", "price", "transmission", "mileage", "fuelType", "engineSize"]

def overlap(sub_name, brand):
    sub = read_clean(DATA_RAW / f"{sub_name}.csv")
    main = df.loc[df["brand"] == brand, key].drop_duplicates()
    m = sub.merge(main, on=key, how="left", indicator=True)
    print(f"{sub_name}: {len(sub)} baris, "
          f"{(m['_merge'] == 'both').mean():.1%} sudah ada di {brand}")

overlap("cclass", "merc")
overlap("focus", "ford")

cclass: 3899 baris, 93.2% sudah ada di merc
focus: 5454 baris, 85.6% sudah ada di ford


In [17]:
# 1. Tahun tidak wajar
print("year > 2020:", (df["year"] > 2020).sum())
print(df.loc[df["year"] > 2020, ["brand", "model", "year", "price", "mileage"]].head(10))
print("\nyear < 2000:", (df["year"] < 2000).sum())
print(df.loc[df["year"] < 2000, ["brand", "model", "year", "price", "mileage"]]
        .sort_values("year").head(10))

# 2. engineSize == 0
zero_eng = df[df["engineSize"] == 0]
print("\nengineSize == 0 per fuelType:\n", zero_eng["fuelType"].value_counts())
print("\nengineSize == 0 per brand:\n", zero_eng["brand"].value_counts())

# 3. mpg ekstrem
print("mpg > 100:", (df["mpg"] > 100).sum())
print(df.loc[df["mpg"] > 100, ["brand", "model", "year", "fuelType", "mpg"]]
        .head(10).to_string())
print("\nmpg < 10:", (df["mpg"] < 10).sum())
print(df.loc[df["mpg"] < 10, ["brand", "model", "year", "fuelType", "mpg"]]
        .head(10).to_string())

# 4. tax == 0
print("\ntax == 0:", (df["tax"] == 0).sum())
print(df.loc[df["tax"] == 0, "fuelType"].value_counts().to_string())

year > 2020: 1
      brand   model  year  price  mileage
39175  ford  Fiesta  2060   6495    54807

year < 2000: 22
          brand     model  year  price  mileage
81235  vauxhall    Zafira  1970  10495    37357
56346      merc   M Class  1970  24999    14000
20293       bmw  5 Series  1996   5995    36000
38327      ford    Escort  1996   3000    50000
56186      merc  SL CLASS  1997   9995    76000
76255  vauxhall     Astra  1997    950    89233
20509       bmw        Z3  1997   3950    49000
10427      audi        A8  1997   4650   122000
56082      merc       SLK  1998   1990   113557
56068      merc   S Class  1998  19990    43534

engineSize == 0 per fuelType:
 fuelType
Petrol      163
Diesel       69
Hybrid       38
Electric      2
Other         1
Name: count, dtype: int64

engineSize == 0 per brand:
 brand
audi        57
ford        51
bmw         47
hyundi      47
vauxhall    33
vw          15
merc        12
toyota       6
skoda        5
Name: count, dtype: int64
mpg > 100: 57

## 4. Keputusan pembersihan

Ringkasan keputusan pembersihan beserta alasan dan asumsinya:

| Keputusan | Alasan / asumsi |
|---|---|
| Pakai 9 file merek; buang `cclass`, `focus`, dan dua file `unclean` | 93,2% baris `cclass` dan 85,6% baris `focus` sudah ada di file merek; kolomnya berbeda |
| Hapus 1.475 duplikat | Asumsi: baris identik di semua kolom = listing ganda (dataset tidak punya kolom ID) |
| Pertahankan `year` 1995-2020 | 2060 jelas salah input; 1970 tidak cocok dengan model dan harganya |
| Buang `engineSize` 0 pada mobil non-listrik | Dianggap data hilang; mobil listrik dipertahankan |
| Buang `mpg` < 10 (33 baris) | Dugaan salah input (contoh yang terlihat semuanya BMW hybrid) |
| Pertahankan `mpg` > 100 dan `tax` = 0 | Dugaan sah (plug-in hybrid, pajak £0); asumsi dari pengetahuan umum |
| Seragamkan `SL Class` dan `X Class` | Kosmetik |
| Model langka (< 10 baris) ditangani di pipeline | 133 baris (0,14%); `OneHotEncoder(min_frequency=10)` dipakai saat pemodelan |

**Hasil:** 1.776 baris dibuang (1,79%), tersisa 97.411 baris.

In [18]:
df_clean = df.copy()
log = [("data gabungan awal", len(df_clean))]

# 1. Duplikat (asumsi: baris identik di semua kolom = listing ganda,
#    karena dataset tidak punya kolom ID)
df_clean = df_clean.drop_duplicates()
log.append(("hapus duplikat", len(df_clean)))

# 2. Tahun di luar rentang wajar (1970 dan 2060 = salah input)
df_clean = df_clean[df_clean["year"].between(1995, 2020)]
log.append(("tahun 1995-2020", len(df_clean)))

# 3. engineSize 0 pada mobil non-listrik = data hilang
bad_engine = (df_clean["engineSize"] == 0) & (df_clean["fuelType"] != "Electric")
df_clean = df_clean[~bad_engine]
log.append(("buang engineSize 0 non-listrik", len(df_clean)))

# 4. mpg < 10 tidak masuk akal (dugaan salah input/satuan)
df_clean = df_clean[df_clean["mpg"] >= 10].reset_index(drop=True)
log.append(("buang mpg < 10", len(df_clean)))

for step, n in log:
    print(f"{step:35} {n:>7}")
removed = len(df) - len(df_clean)
print(f"\nTotal dibuang: {removed} ({removed / len(df):.2%})")

data gabungan awal                    99187
hapus duplikat                        97712
tahun 1995-2020                       97709
buang engineSize 0 non-listrik        97444
buang mpg < 10                        97411

Total dibuang: 1776 (1.79%)


## 5. Standarisasi nama model dan model langka

In [19]:
# --- Cek variasi penulisan model (abaikan huruf besar/kecil, spasi, tanda hubung)
tmp = df_clean[["brand", "model"]].drop_duplicates().copy()
tmp["norm"] = tmp["model"].str.lower().str.replace(r"[\s\-]", "", regex=True)
n_var = tmp.groupby(["brand", "norm"])["model"].transform("nunique")
print("Model dengan variasi penulisan ganda:", (n_var > 1).sum())

# --- Apakah ada nama model yang dipakai lebih dari satu merek?
multi = df_clean.groupby("model")["brand"].nunique()
print("Model yang muncul di >1 merek:", (multi > 1).sum())

# --- Rapikan penulisan (kosmetik)
df_clean["model"] = df_clean["model"].replace(
    {"SL CLASS": "SL Class", "X-CLASS": "X Class"}
)

# --- Model langka (ditangani di pipeline: OneHotEncoder min_frequency=10)
counts = df_clean.groupby(["brand", "model"]).size()
rare = counts[counts < 10]
print("Kombinasi brand-model:", len(counts))
print("Kombinasi < 10 baris :", len(rare),
      f"({rare.sum()} baris, {rare.sum() / len(df_clean):.2%} dari data)")

Model dengan variasi penulisan ganda: 0
Model yang muncul di >1 merek: 0
Kombinasi brand-model: 194
Kombinasi < 10 baris : 36 (133 baris, 0.14% dari data)


## 6. Validasi dan penyimpanan data bersih

In [20]:
# Sanity check: akan error kalau ada langkah yang terlewat
assert df_clean.isna().sum().sum() == 0
assert not df_clean.duplicated().any()
assert df_clean["year"].between(1995, 2020).all()
assert (df_clean["mpg"] >= 10).all()
assert not ((df_clean["engineSize"] == 0) & (df_clean["fuelType"] != "Electric")).any()
assert not df_clean["model"].isin(["SL CLASS", "X-CLASS"]).any()
assert len(df_clean) == 97411   # angka dari run saat ini; deteksi sel yang tak sengaja terhapus

df_clean.to_csv("../data/processed/cars_clean.csv", index=False)
print("Tersimpan:", df_clean.shape)

Tersimpan: (97411, 10)
